# Sweep 16 Champion — MIMIC-IV SOTA Cohort (~9K patients)

**Cohort**: Strict ICD-9 admission-level filter — only admissions with NO ICD-10 codes.
~9K patients, directly comparable to HI-DR, VITA, CIDGMed, ARMR published numbers.

**Architecture**: Champion from sweeps 11a–15b + Sweep 17 Gap D (hdim=128)

| Cell | Purpose | Time |
|------|---------|------|
| 1 | Install | 1 min |
| 2 | Kaggle setup | 2 min |
| 3 | Config + helpers | — |
| 4 | Smoke test (1 epoch) | 2 min |
| 5 | alpha=0.0 (5 seeds) | ~1.5h |
| 6 | alpha=0.2 (5 seeds) | ~1.5h |
| 7 | alpha=0.5 (5 seeds) | ~1.5h |
| 8 | Run log | — |
| 9 | Results table | — |
| 10 | Export zip | — |

In [ ]:
import torch
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
!pip install -q torch_geometric
!pip install -q pyyaml pandas numpy scikit-learn


In [ ]:
import os, sys, glob

if os.path.exists("/kaggle"):
    print("Running on Kaggle")
    os.chdir("/kaggle/working")
    os.system("rm -rf ./data ./src")
    os.makedirs("data/processed", exist_ok=True)
    os.makedirs("data/embeddings", exist_ok=True)

    train_paths = glob.glob("/kaggle/input/**/train.py", recursive=True)
    if not train_paths:
        raise FileNotFoundError("train.py not found in /kaggle/input")
    src_dir = os.path.dirname(train_paths[0])
    os.system(f"cp -r {src_dir} /kaggle/working/src")
    sys.path.append("/kaggle/working/src")

    # Processed files — anchor on cohort_mimic4_sota.pkl
    for find_file in ["cohort_mimic4_sota.pkl"]:
        found = glob.glob(f"/kaggle/input/**/{find_file}", recursive=True)
        if found:
            processed_dir = os.path.dirname(found[0])
            print(f"Found processed dir at: {processed_dir}")
            for root, dirs, files in os.walk(processed_dir):
                rel = os.path.relpath(root, processed_dir)
                target_dir = os.path.join("./data/processed", rel) if rel != "." else "./data/processed"
                os.makedirs(target_dir, exist_ok=True)
                for fname in files:
                    src = os.path.join(root, fname)
                    link = os.path.join(target_dir, fname)
                    if not os.path.exists(link):
                        try:
                            os.symlink(src, link)
                        except OSError:
                            os.system(f"cp '{src}' '{link}'")
            break

    emb_paths = glob.glob("/kaggle/input/**/code_embeddings_mimic4_sota.pt", recursive=True)
    if not emb_paths:
        raise FileNotFoundError("code_embeddings_mimic4_sota.pt not found")
    emb_dir = os.path.dirname(emb_paths[0])
    for fpath in glob.glob(f"{emb_dir}/*"):
        fname = os.path.basename(fpath)
        link = f"./data/embeddings/{fname}"
        if not os.path.exists(link):
            os.symlink(fpath, link)

    note_paths = glob.glob("/kaggle/input/**/note_embeddings_mimic4_sota.pkl", recursive=True)
    if note_paths:
        note_src = note_paths[0]
        note_link = "./data/processed/note_embeddings_mimic4_sota.pkl"
        if not os.path.exists(note_link):
            os.symlink(note_src, note_link)
        print(f"Note embeddings linked: {note_src}")
    else:
        print("WARNING: note_embeddings_mimic4_sota.pkl not found")

    # Patch 1: registry.py
    reg_path = "/kaggle/working/src/model/registry.py"
    if os.path.exists(reg_path):
        with open(reg_path, "r") as rf:
            content = rf.read()
        if "import inspect" not in content:
            old = "        return cls(**kwargs)"
            new = (
                "        import inspect\n"
                "        sig = inspect.signature(cls.__init__)\n"
                "        has_var_keyword = any(p.kind == inspect.Parameter.VAR_KEYWORD "
                "for p in sig.parameters.values())\n"
                "        if has_var_keyword:\n"
                "            filtered_kwargs = kwargs\n"
                "        else:\n"
                "            filtered_kwargs = {k: v for k, v in kwargs.items() if k in sig.parameters}\n"
                "        return cls(**filtered_kwargs)"
            )
            if old in content:
                content = content.replace(old, new)
                with open(reg_path, "w") as wf:
                    wf.write(content)
                print("  registry.py patched")

    # Patch 2: drug_gnn.py
    gnn_path = "/kaggle/working/src/model/graph_encoders/drug_gnn.py"
    if os.path.exists(gnn_path):
        with open(gnn_path, "r") as rf:
            content = rf.read()
        if "ehr_adj" not in content:
            old = "        use_lab_nodes: bool = False,\n        lab_embeddings: torch.Tensor | None = None,   # (num_labs, 768)\n    ):"
            new = "        use_lab_nodes: bool = False,\n        lab_embeddings: torch.Tensor | None = None,   # (num_labs, 768)\n        ehr_adj=None,\n        **kwargs,\n    ):"
            if old in content:
                content = content.replace(old, new)
                with open(gnn_path, "w") as wf:
                    wf.write(content)
                print("  drug_gnn.py patched")

    # Patch 3: dcma_decoder.py
    dcma_path = "/kaggle/working/src/model/decoders/dcma_decoder.py"
    if os.path.exists(dcma_path):
        with open(dcma_path, "r") as rf:
            content = rf.read()
        if "note_proj" not in content:
            content = content.replace(
                "    def __init__(self, hidden_dim: int = 256, dropout: float = 0.3, **kwargs):\n"
                "        super().__init__()\n"
                "        self.attn = DrugModalityAttention(hidden_dim, dropout)",
                "    def __init__(self, hidden_dim: int = 256, dropout: float = 0.3,\n"
                "                 note_repr_dim: int = 768, lab_repr_dim: int = 400, **kwargs):\n"
                "        super().__init__()\n"
                "        self.attn = DrugModalityAttention(hidden_dim, dropout)\n"
                "        self.hidden_dim = hidden_dim\n"
                "        self.note_proj = nn.Linear(note_repr_dim, hidden_dim)\n"
                "        self.lab_proj  = nn.Linear(lab_repr_dim,  hidden_dim)",
            )
            with open(dcma_path, "w") as wf:
                wf.write(content)
            print("  dcma_decoder.py patched")

print("Setup done:", os.getcwd())


In [ ]:
import yaml, subprocess, gc, json, numpy as np, os as _os
from pathlib import Path

BASE_CONFIG = "src/config.yaml"

def make_config(output_path, model_overrides=None, training_overrides=None,
                preprocessing_overrides=None, smoke_epochs=None):
    with open(BASE_CONFIG, "r") as f:
        cfg = yaml.safe_load(f)
    if model_overrides:
        for k, v in model_overrides.items():
            cfg["model"][k] = v
    if training_overrides:
        for k, v in training_overrides.items():
            cfg["training"][k] = v
    if preprocessing_overrides:
        for k, v in preprocessing_overrides.items():
            cfg["preprocessing"][k] = v
    if smoke_epochs is not None:
        cfg["training"]["epochs"] = smoke_epochs
    with open(output_path, "w") as f:
        yaml.dump(cfg, f, default_flow_style=False)
    return str(output_path)

# Champion architecture locked from sweeps 11a-15b + 17D
CHAMPION_MODEL = {
    "hidden_dim":         128,
    "ar_max_seq_len":     20,
    "note_proj_dim":      64,
    "graph_encoder_type": "drug_gnn",
    "graph_layer_type":   "gcn",
    "hgt_layers":         2,
    "encoder_type":       "imdr_infused",
    "predictor_type":     "heidr",
}
CHAMPION_TRAINING = {
    "bce_weight":           0.3,
    "soft_jaccard_weight":  1.5,
    "margin_weight":        0.05,
}
BACKBONE_FLAGS = (
    "--device cuda "
    "--visit_level_training "
    "--fusion_strategy film "
    "--aggregator_type last "
    "--mimic_version 4 --mimic4_sota "
    "--processed_dir data/processed "
    "--embeddings_dir data/embeddings "
)

# Lab file — prefer sota-specific, fall back to generic 200labs name
_lab_candidates = [
    "data/processed/lab_vectors_mimic4_sota.pkl",
    "data/processed/lab_vectors_200labs.pkl",
]
_lab_file = next((p for p in _lab_candidates if _os.path.exists(p)), _lab_candidates[0])
print(f"Lab file: {_lab_file}")
LAB_FLAGS = f"--lab_pkl {_lab_file} --num_labs 200"
CHAMPION_PREP = {"lab_dim": 400}

SEEDS      = [42, 123, 456, 789, 1024]
DDI_ALPHAS = [0.0, 0.2, 0.5]

reports_dir = Path("experiment_reports/sweep_16_champion_mimic4sota")
results_log = []

def alpha_tag(a):
    return str(a).replace(".", "p")

def run_champion(ddi_alpha):
    import torch
    atag = alpha_tag(ddi_alpha)
    cfgp = f"s16m4s_champion_alpha{atag}.yaml"
    make_config(cfgp, model_overrides=CHAMPION_MODEL,
                training_overrides=CHAMPION_TRAINING,
                preprocessing_overrides=CHAMPION_PREP)
    print(f"\n{'='*65}")
    print(f"  Champion (MIMIC-IV SOTA ~8.8K) -- ddi_alpha={ddi_alpha}  ({len(SEEDS)} seeds)")
    print(f"{'='*65}")
    for seed in SEEDS:
        run_name = f"champion_mimic4sota_alpha{atag}_seed{seed}"
        run_dir  = reports_dir / run_name
        run_dir.mkdir(parents=True, exist_ok=True)
        if list(run_dir.glob("result_*.json")):
            print(f"  [SKIP] {run_name} — already done")
            results_log.append(f"SKIP: {run_name}")
            continue
        log_path = run_dir / "training_log.txt"
        cmd = (f"python -u src/train.py --config {cfgp} "
               f"{BACKBONE_FLAGS} {LAB_FLAGS} "
               f"--ddi_alpha {ddi_alpha} "
               f"--seed {seed} --results_dir {run_dir}")
        print(f"\n  --- {run_name} ---")
        try:
            with open(log_path, "w") as lf:
                proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                                        stderr=subprocess.STDOUT, text=True)
                for line in proc.stdout:
                    print(line, end="")
                    lf.write(line)
                proc.wait()
            status = "SUCCESS" if proc.returncode == 0 else f"FAILED ({proc.returncode})"
            results_log.append(f"{status}: {run_name}")
        except Exception as e:
            results_log.append(f"CRASH: {run_name}: {e}")
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

print(f"Config ready. reports_dir = {reports_dir}")
print(f"Seeds: {SEEDS}  |  DDI alphas: {DDI_ALPHAS}  ->  {len(SEEDS)*len(DDI_ALPHAS)} total runs")
print("Cohort: MIMIC-IV SOTA (~8.8K patients, ICU-only, Carmen/ARMR/VITA/HI-DR benchmark)")

In [ ]:
# Smoke test: 1 epoch x alpha=0.0 before committing to full runs
Path("smoke_s16m4s").mkdir(exist_ok=True)
cfg_path = "s16m4s_smoke.yaml"
make_config(cfg_path,
            model_overrides=CHAMPION_MODEL,
            training_overrides=CHAMPION_TRAINING,
            preprocessing_overrides=CHAMPION_PREP,
            smoke_epochs=1)
cmd = (f"python -u src/train.py --config {cfg_path} "
       f"{BACKBONE_FLAGS} {LAB_FLAGS} "
       f"--ddi_alpha 0.0 "
       f"--seed 42 --results_dir smoke_s16m4s/champion")
print("SMOKE: running 1 epoch...")
proc = subprocess.run(cmd, shell=True, capture_output=True, text=True)
for ln in (proc.stdout + proc.stderr).strip().split("\n")[-8:]: print(ln)
status = "PASS" if proc.returncode == 0 else f"FAIL ({proc.returncode})"
print(f">>> {status}")
if "FAIL" in status:
    print("Full stderr:")
    print(proc.stderr[-3000:])
    raise RuntimeError("Smoke test failed — fix before running full sweep")


In [ ]:
# Cell 5 — alpha=0.0  baseline, no DDI loss  (~1.5h for 5 seeds on ~9K cohort)
run_champion(ddi_alpha=0.0)


In [ ]:
# Cell 6 — alpha=0.2  balanced: small Jaccard cost, meaningful DDI reduction  (~1.5h)
run_champion(ddi_alpha=0.2)


In [ ]:
# Cell 7 — alpha=0.5  safety-first: targets DDI rate below 0.06  (~1.5h)
run_champion(ddi_alpha=0.5)


In [ ]:
print(f"\nRun log ({len(results_log)} entries):")
for e in results_log: print(" ", e)


In [ ]:
import json, numpy as np
from pathlib import Path

reports_dir = Path("experiment_reports/sweep_16_champion_mimic4sota")

def alpha_tag(a): return str(a).replace(".", "p")

SEEDS      = [42, 123, 456, 789, 1024]
DDI_ALPHAS = [0.0, 0.2, 0.5]

print("=" * 75)
print("  MIRROR Champion — MIMIC-IV SOTA (~9K, strict ICD-9) — 3-alpha sweep")
print("=" * 75)

for alpha in DDI_ALPHAS:
    atag = alpha_tag(alpha)
    jacs, ddis = [], []
    for seed in SEEDS:
        run_name = f"champion_mimic4sota_alpha{atag}_seed{seed}"
        run_dir  = reports_dir / run_name
        files    = list(run_dir.glob("result_*.json")) if run_dir.exists() else []
        if not files:
            continue
        r = json.loads(files[0].read_text())
        tm = r.get("test_metrics", {})
        jacs.append(tm.get("jaccard", float("nan")))
        ddis.append(tm.get("ddi_rate", float("nan")))

    if jacs:
        print(f"\n  alpha={alpha:3.1f}  ({len(jacs)}/5 seeds done)")
        print(f"    Jaccard : {np.nanmean(jacs):.4f} ± {np.nanstd(jacs):.4f}  "
              f"[{np.nanmin(jacs):.4f} – {np.nanmax(jacs):.4f}]")
        print(f"    DDI rate: {np.nanmean(ddis):.4f} ± {np.nanstd(ddis):.4f}  "
              f"{'✓ below 0.06' if np.nanmean(ddis) < 0.06 else 'above 0.06'}")
    else:
        print(f"\n  alpha={alpha:3.1f}  — no results yet")

print("\n  Note: compare Jaccard directly to HI-DR/VITA/CIDGMed/ARMR MIMIC-IV numbers")


In [ ]:
import shutil, os
from pathlib import Path

reports_dir = Path("experiment_reports/sweep_16_champion_mimic4sota")
zip_path = "sweep_16_champion_mimic4sota_results.zip"

if reports_dir.exists():
    shutil.make_archive(zip_path.replace(".zip", ""), "zip", reports_dir)
    size_mb = os.path.getsize(zip_path) / 1e6
    print(f"Exported: {zip_path} ({size_mb:.1f} MB)")
else:
    print("No results directory found yet.")
